Perfect 👌 — let’s wire this up cleanly so that your **agents actually hit the running MCP servers**.
I’ll give you the **weather client**, **pollution client**, and the **necessary changes in factory + tools**.

---

## 1️⃣ `mcp_clients/weather_client.py`

```python
# mcp_clients/weather_client.py
import asyncio
from mcp.client.stdio import StdioClient

async def get_weather(location: str) -> str:
    """Call the Weather MCP server tool"""
    client = StdioClient(command=["python", "mcp_servers/weather_server.py"])
    async with client:
        result = await client.call("get_weather", {"location": location})
        return result.content

if __name__ == "__main__":
    async def main():
        res = await get_weather("Paris")
        print("Weather in Paris:", res)

    asyncio.run(main())
```

---

## 2️⃣ `mcp_clients/pollution_client.py`

```python
# mcp_clients/pollution_client.py
import asyncio
from mcp.client.stdio import StdioClient

async def get_pollution(location: str) -> str:
    """Call the Pollution MCP server tool"""
    client = StdioClient(command=["python", "mcp_servers/pollution_server.py"])
    async with client:
        result = await client.call("get_pollution", {"location": location})
        return result.content

if __name__ == "__main__":
    async def main():
        res = await get_pollution("Delhi")
        print("Pollution in Delhi:", res)

    asyncio.run(main())
```

---

## 3️⃣ `tools/weather_tools.py`

```python
# tools/weather_tools.py
import asyncio
from mcp_clients.weather_client import get_weather

async def get_city_weather(city: str) -> str:
    return await get_weather(city)
```

---

## 4️⃣ `tools/pollution_tools.py`

```python
# tools/pollution_tools.py
import asyncio
from mcp_clients.pollution_client import get_pollution

async def get_city_pollution(city: str) -> str:
    return await get_pollution(city)
```

---

## 5️⃣ Update **AgentFactory**

```python
# agents/agent_factory.py
from config.settings import AGENT_CONFIG
from tools import weather_tools, pollution_tools

class AgentFactory:
    def __init__(self):
        self.agents = {}

    def build_agent(self, agent_name: str):
        if agent_name not in AGENT_CONFIG:
            raise ValueError(f"Unknown agent: {agent_name}")

        config = AGENT_CONFIG[agent_name]

        if agent_name == "weather":
            async def run(task: dict):
                city = task.get("city")
                if not city:
                    return "Weather agent: no city provided"
                return await weather_tools.get_city_weather(city)

        elif agent_name == "pollution":
            async def run(task: dict):
                city = task.get("city")
                if not city:
                    return "Pollution agent: no city provided"
                return await pollution_tools.get_city_pollution(city)

        else:
            async def run(task: dict):
                return f"{agent_name} agent not implemented"

        self.agents[agent_name] = run
        return run

    def get_agent(self, agent_name: str):
        return self.agents.get(agent_name) or self.build_agent(agent_name)
```

---

## 6️⃣ `main.py` (only change = `run_async` stays the same)

Now when you run:

```bash
python main.py
```

👉 The graph will:

* Route prompt → parent agent (LLM router)
* Call **Weather agent → Weather MCP Client → Weather MCP Server → wttr.in**
* Or **Pollution agent → Pollution MCP Client → Pollution MCP Server → static AQI data**

---

✅ With this, you **don’t need to manually start the MCP servers** — because `StdioClient` starts them automatically when called.

---

Do you want me to also update your `mcp_servers/weather_server.py` and `pollution_server.py` so they’re aligned with **`fastmcp 2.12.2`** (no `Response` import issue)?
